# Generating Training Data
The data I have is spread around multiple CSV files. This notebook will be used to combine the data from the multiple files into a (m,22,9) as the dataset features for players and a (m, 2, 1) array with the target labels. The steps will include:

1. Getting matches played no later than 1970
2. For each match
    i. Get players from home team
    ii. Get players from away team
    iii. Combine two lists into players
    iv. For each player
        a. Get statistics at that point
    v. Store players in feature array
    vi. Store match result in target array
3. Save features and target array

Depending on the number of matches, I will use batching.

In [69]:
# Imports
import pandas as pd
import numpy as np
import numpy.typing as npt
import math
from datetime import datetime

In [2]:
# Get matches after 1970
matches = pd.read_csv("matches.csv")
matches.head()

,key_id,tournament_id,tournament_name,match_id,match_name,stage_name,group_name,group_stage,knockout_stage,replayed,...,away_team_score_margin,extra_time,penalty_shootout,score_penalties,home_team_score_penalties,away_team_score_penalties,result,home_team_win,away_team_win,draw
0,1,WC-1930,1930 FIFA World Cup,M-1930-01,France v Mexico,group stage,Group 1,1,0,0,...,-3,0,0,0-0,0,0,home team win,1,0,0
1,2,WC-1930,1930 FIFA World Cup,M-1930-02,United States v Belgium,group stage,Group 4,1,0,0,...,-3,0,0,0-0,0,0,home team win,1,0,0
2,3,WC-1930,1930 FIFA World Cup,M-1930-03,Yugoslavia v Brazil,group stage,Group 2,1,0,0,...,-1,0,0,0-0,0,0,home team win,1,0,0
3,4,WC-1930,1930 FIFA World Cup,M-1930-04,Romania v Peru,group stage,Group 3,1,0,0,...,-2,0,0,0-0,0,0,home team win,1,0,0
4,5,WC-1930,1930 FIFA World Cup,M-1930-05,Argentina v France,group stage,Group 1,1,0,0,...,-1,0,0,0-0,0,0,home team win,1,0,0


In [3]:
# Explore what's in the dataset
matches.columns

Index(['key_id', 'tournament_id', 'tournament_name', 'match_id', 'match_name',
       'stage_name', 'group_name', 'group_stage', 'knockout_stage', 'replayed',
       'replay', 'match_date', 'match_time', 'stadium_id', 'stadium_name',
       'city_name', 'country_name', 'home_team_id', 'home_team_name',
       'home_team_code', 'away_team_id', 'away_team_name', 'away_team_code',
       'score', 'home_team_score', 'away_team_score', 'home_team_score_margin',
       'away_team_score_margin', 'extra_time', 'penalty_shootout',
       'score_penalties', 'home_team_score_penalties',
       'away_team_score_penalties', 'result', 'home_team_win', 'away_team_win',
       'draw'],
      dtype='str')

In [105]:
reduced_matches = matches[['match_id', 'tournament_id', 'home_team_id', 'away_team_id', 'home_team_score', 'away_team_score', 'match_date']]
reduced_matches.head()

,match_id,tournament_id,home_team_id,away_team_id,home_team_score,away_team_score,match_date
0,M-1930-01,WC-1930,T-28,T-44,4,1,1930-07-13
1,M-1930-02,WC-1930,T-79,T-06,3,0,1930-07-13
2,M-1930-03,WC-1930,T-83,T-09,2,1,1930-07-14
3,M-1930-04,WC-1930,T-58,T-54,3,1,1930-07-14
4,M-1930-05,WC-1930,T-03,T-28,1,0,1930-07-15


In [106]:
# Extract year from tournamet_id
reduced_matches['year'] = reduced_matches['tournament_id'].apply(lambda x: int(x.split("-")[-1]))
reduced_matches.head()

,match_id,tournament_id,home_team_id,away_team_id,home_team_score,away_team_score,match_date,year
0,M-1930-01,WC-1930,T-28,T-44,4,1,1930-07-13,1930
1,M-1930-02,WC-1930,T-79,T-06,3,0,1930-07-13,1930
2,M-1930-03,WC-1930,T-83,T-09,2,1,1930-07-14,1930
3,M-1930-04,WC-1930,T-58,T-54,3,1,1930-07-14,1930
4,M-1930-05,WC-1930,T-03,T-28,1,0,1930-07-15,1930


In [107]:
reduced_matches = reduced_matches.drop('tournament_id', axis=1)
reduced_matches.head()

,match_id,home_team_id,away_team_id,home_team_score,away_team_score,match_date,year
0,M-1930-01,T-28,T-44,4,1,1930-07-13,1930
1,M-1930-02,T-79,T-06,3,0,1930-07-13,1930
2,M-1930-03,T-83,T-09,2,1,1930-07-14,1930
3,M-1930-04,T-58,T-54,3,1,1930-07-14,1930
4,M-1930-05,T-03,T-28,1,0,1930-07-15,1930


In [108]:
matches_later_1970 =reduced_matches[reduced_matches['year'] >= 1970]
matches_later_1970.head()

,match_id,home_team_id,away_team_id,home_team_score,away_team_score,match_date,year
200,M-1970-01,T-44,T-69,0,0,1970-05-31,1970
201,M-1970-02,T-80,T-38,2,0,1970-06-02,1970
202,M-1970-03,T-27,T-58,1,0,1970-06-02,1970
203,M-1970-04,T-54,T-10,3,2,1970-06-02,1970
204,M-1970-05,T-06,T-26,3,0,1970-06-03,1970


In [109]:
matches_later_1970.shape

(700, 7)

In [110]:
# Import squad details
squad = pd.read_csv("squads.csv")
squad.head()

,key_id,tournament_id,tournament_name,team_id,team_name,team_code,player_id,family_name,given_name,shirt_number,position_name,position_code
0,1,WC-1930,1930 FIFA World Cup,T-03,Argentina,ARG,P-01578,Bossio,Ángel,0,goal keeper,GK
1,2,WC-1930,1930 FIFA World Cup,T-03,Argentina,ARG,P-02109,Botasso,Juan,0,goal keeper,GK
2,3,WC-1930,1930 FIFA World Cup,T-03,Argentina,ARG,P-06974,Cherro,Roberto,0,forward,FW
3,4,WC-1930,1930 FIFA World Cup,T-03,Argentina,ARG,P-07975,Chividini,Alberto,0,defender,DF
4,5,WC-1930,1930 FIFA World Cup,T-03,Argentina,ARG,P-06464,Della Torre,José,0,defender,DF


In [111]:
squad['year'] = squad['tournament_id'].apply(lambda x: int(x.split("-")[-1]))
squad.head()

,key_id,tournament_id,tournament_name,team_id,team_name,team_code,player_id,family_name,given_name,shirt_number,position_name,position_code,year
0,1,WC-1930,1930 FIFA World Cup,T-03,Argentina,ARG,P-01578,Bossio,Ángel,0,goal keeper,GK,1930
1,2,WC-1930,1930 FIFA World Cup,T-03,Argentina,ARG,P-02109,Botasso,Juan,0,goal keeper,GK,1930
2,3,WC-1930,1930 FIFA World Cup,T-03,Argentina,ARG,P-06974,Cherro,Roberto,0,forward,FW,1930
3,4,WC-1930,1930 FIFA World Cup,T-03,Argentina,ARG,P-07975,Chividini,Alberto,0,defender,DF,1930
4,5,WC-1930,1930 FIFA World Cup,T-03,Argentina,ARG,P-06464,Della Torre,José,0,defender,DF,1930


In [112]:
reduced_squad = squad[squad['year'] >= 1970]
reduced_squad.head(n=11)

,key_id,tournament_id,tournament_name,team_id,team_name,team_code,player_id,family_name,given_name,shirt_number,position_name,position_code,year
2594,2595,WC-1970,1970 FIFA World Cup,T-06,Belgium,BEL,P-02581,Piot,Christian,1,goal keeper,GK,1970
2595,2596,WC-1970,1970 FIFA World Cup,T-06,Belgium,BEL,P-00773,Heylens,Georges,2,defender,DF,1970
2596,2597,WC-1970,1970 FIFA World Cup,T-06,Belgium,BEL,P-02546,Thissen,Jean,3,defender,DF,1970
2597,2598,WC-1970,1970 FIFA World Cup,T-06,Belgium,BEL,P-08103,Dewalque,Nicolas,4,defender,DF,1970
2598,2599,WC-1970,1970 FIFA World Cup,T-06,Belgium,BEL,P-01028,Jeck,Léon,5,defender,DF,1970
2599,2600,WC-1970,1970 FIFA World Cup,T-06,Belgium,BEL,P-06938,Dockx,Jean,6,midfielder,MF,1970
2600,2601,WC-1970,1970 FIFA World Cup,T-06,Belgium,BEL,P-00526,Semmeling,Léon,7,midfielder,MF,1970
2601,2602,WC-1970,1970 FIFA World Cup,T-06,Belgium,BEL,P-02611,Van Moer,Wilfried,8,midfielder,MF,1970
2602,2603,WC-1970,1970 FIFA World Cup,T-06,Belgium,BEL,P-06209,Devrindt,Johan,9,forward,FW,1970
2603,2604,WC-1970,1970 FIFA World Cup,T-06,Belgium,BEL,P-06673,Van Himst,Paul,10,forward,FW,1970


In [113]:
reduced_squad.columns

Index(['key_id', 'tournament_id', 'tournament_name', 'team_id', 'team_name',
       'team_code', 'player_id', 'family_name', 'given_name', 'shirt_number',
       'position_name', 'position_code', 'year'],
      dtype='str')

In [114]:
players = pd.read_csv('players.csv')
players.head()

,key_id,player_id,family_name,given_name,birth_date,goal_keeper,defender,midfielder,forward,count_tournaments,list_tournaments,player_wikipedia_link
0,1,P-05074,A'Court,Alan,1934-09-30,0,0,0,1,1,1958,https://en.wikipedia.org/wiki/Alan_A%27Court
1,2,P-00942,Abadzhiev,Stefan,1934-07-03,0,0,0,1,1,1966,https://en.wikipedia.org/wiki/Stefan_Abadzhiev
2,3,P-03051,Abalo,Jean-Paul,1975-06-26,0,1,0,0,1,2006,https://en.wikipedia.org/wiki/Jean-Paul_Abalo
3,4,P-03371,Abanda,Patrice,1978-08-03,0,1,0,0,1,1998,https://en.wikipedia.org/wiki/Patrice_Abanda
4,5,P-04977,Abate,Ignazio,1986-11-12,0,1,0,0,1,2014,https://en.wikipedia.org/wiki/Ignazio_Abate


In [115]:
players.dtypes

key_id                   int64
player_id                  str
family_name                str
given_name                 str
birth_date                 str
goal_keeper              int64
defender                 int64
midfielder               int64
forward                  int64
count_tournaments        int64
list_tournaments           str
player_wikipedia_link      str
dtype: object

In [124]:
statistics = pd.read_csv("data/players_2026-05-23 19:03:04.935108 copy.csv")
statistics.head()

,id,appearances,team,goals,from,to
0,P-05074,354.0,Liverpool,61.0,1952.0,1964.0
1,P-05074,50.0,Tranmere Rovers,11.0,1964.0,1966.0
2,P-05074,0.0,Norwich City,0.0,1966.0,1967.0
3,P-00942,254.0,Levski Sofia,37.0,1953.0,1968.0
4,P-00942,NaN,Wiesbaden,NaN,1968.0,1970.0


In [150]:
def get_players_statistics(player_id: str, match_date: str):
    """ 
    Return statistics for a player:
        1. age
        2. forward
        3. middlefielder
        4. defense
        5. goalkeeper
        6. num_tournaments
        7. appearances
        8. goals
        9. average time per team
    
    Args:
        player_id (str): id of player
        match_date (str): day match was played
        
    Returns:
        x_i (ndarray): (1,9) array with statistics of each player
    """    
    player = players[players['player_id'] == player_id]
    if len(player) < 1:
        return np.array([
            [0,0,0,0,0,0,0,0,0]
        ])
    
    # Age of player
    raw_birthdate = player['birth_date'].iloc[0]
    birthdate = datetime.strptime(raw_birthdate, "%Y-%m-%d")
    tournament_date = datetime.strptime(match_date, "%Y-%m-%d")
    match_year = tournament_date.year
    difference_in_days = (tournament_date - birthdate).days
    age = int(difference_in_days / 365.25)
    
    forward = player['forward'].iloc[0]
    middlefielder = player['midfielder'].iloc[0]
    defense = player['defender'].iloc[0]
    goalkeeper = player['goal_keeper'].iloc[0]
    
    raw_tournaments = player['list_tournaments'].iloc[0].split(",")
    tournaments = [int(t) < match_year for t in raw_tournaments]
    num_tournaments = sum(tournaments)
    
    appearances = 0
    goals = 0
    average_time_per_team = 0
    num_teams = 0
    player_statistics = statistics[statistics['id'] == player_id]
    
    if len(player_statistics) < 1:
        return np.array(
            [[age, forward, middlefielder, defense, goalkeeper, num_tournaments, appearances, goals, average_time_per_team]]
        )
    
    # Get stats as per tournament year
    for stat in player_statistics.iterrows():
        s_from = stat[1]["from"]
        s_to = match_year if np.isnan(stat[1]["to"]) else stat[1]["to"]
        s_appearance = 0 if np.isnan(stat[1]['appearances']) else stat[1]['appearances']
        s_goals = 0 if np.isnan(stat[1]['goals']) else stat[1]['goals']
        
        
        if s_from < match_year and s_to < match_year:
            appearances += s_appearance
            goals += s_goals
            num_teams += 1
            average_time_per_team += s_to - s_from
        elif s_from < match_year:
            total = s_to - s_from
            num_years = match_year - s_from
            appearances += int(s_appearance * (num_years / total))
            goals += int(s_goals * (num_years / total))
            num_teams += 1
            average_time_per_team += num_years
            
            
    if num_teams == 0:
        average_time_per_team = 0
    else:
        average_time_per_team /= num_teams
        
    return np.array([
            [age, forward, middlefielder, defense, goalkeeper, num_tournaments, appearances, goals, average_time_per_team]
    ]
        )
    
player = get_players_statistics("P-02581", "1970-05-31")
print(player.shape)

(1, 9)


In [151]:
def get_world_cup_match_data(match):
    """ 
    Function to get training example from a world cup match.
    
    Args:
        match: (PandasNamedTuple): A row from the matches_later_1970 dataframe
        
    Returns:
        x_i: (ndarray): A (22,9) ndarray that has the statistics of each player in home and away team
        y_i: (ndarray): A (2,1) ndarray showing result of the match
    """
    x_i = np.empty((0,9))
    year = match.year
    home_team_id = match.home_team_id
    away_team_id = match.away_team_id
    match_date = match.match_date
    
    # Get players in home team (Assumes that first squad is first 11 in dataset for a particular team)
    home_team = reduced_squad[(reduced_squad['year'] == year) & (reduced_squad['team_id'] == home_team_id)].head(n=11)
    home_team_player_ids = home_team['player_id']
    if len(home_team_player_ids) != 11:
        raise Exception("Could not get players for home team")
    
    player_ids = home_team_player_ids.tolist()

    # Get players in away team
    away_team = reduced_squad[(reduced_squad['year'] == year) & (reduced_squad['team_id'] == away_team_id)].head(n=11)
    away_team_player_ids = away_team['player_id']
    if len(away_team_player_ids) != 11:
        raise Exception("Could not get away players")
    
    player_ids.extend(away_team_player_ids.tolist())
    
    # For each player
    for player_id in player_ids:
        # Get statistics
        stats = get_players_statistics(player_id=player_id, match_date=match_date)
        # Store in x_i
        x_i = np.append(x_i, stats, axis=0)
    
    
    # Get result
    y_i = np.array([
        [match.home_team_score],
        [match.away_team_score]
    ])
    return (x_i, y_i)

print(get_world_cup_match_data(matches_later_1970.iloc[0]))

(array([[ 26.        ,   0.        ,   0.        ,   0.        ,
          1.        ,   1.        ,   0.        ,   0.        ,
          8.        ],
       [ 26.        ,   0.        ,   0.        ,   1.        ,
          0.        ,   0.        ,   0.        ,   0.        ,
          3.        ],
       [ 28.        ,   0.        ,   0.        ,   1.        ,
          0.        ,   1.        ,   0.        ,   0.        ,
          5.        ],
       [ 27.        ,   0.        ,   0.        ,   1.        ,
          0.        ,   0.        ,   0.        ,   0.        ,
          6.        ],
       [ 23.        ,   0.        ,   0.        ,   1.        ,
          0.        ,   0.        ,   0.        ,   0.        ,
          5.        ],
       [ 27.        ,   0.        ,   0.        ,   1.        ,
          0.        ,   1.        ,   0.        ,   0.        ,
          0.        ],
       [ 22.        ,   0.        ,   1.        ,   0.        ,
          0.        ,   0.   

In [152]:
def create_world_cup_match_dataset():
    """ 
    Creates dataset on world cup matches data and saves into a file
    """
    batch_size = 100
    total_matches = matches_later_1970.shape[0]
    total_batches = math.ceil(total_matches / batch_size)
    X = np.empty((0,22,9))
    Y = np.empty((0,2,1))
    
    for i in range(total_batches):
        start = i * batch_size
        if i == total_batches - 1:
            end = total_matches
        else:
            end = start + batch_size
            
        print(f"Processing batch #{i} which goes from row {start} to {end}")
        
        batch_matches = matches_later_1970.iloc[start:end]
        players_list = []
        results_list = []
        
        players_file_name = f"players_batch_{i}_{datetime.now()}"
        results_file_name = f"results_batch_{i}_{datetime.now()}"
        
        for match in batch_matches.itertuples():
            x_i, y_i = get_world_cup_match_data(match=match)
            players_list.append(x_i)
            results_list.append(y_i)
        
        X_b = np.array(players_list)
        Y_b = np.array(results_list)
        X = np.append(X, X_b, axis=0)
        Y = np.append(Y, Y_b, axis=0)
        
        np.save(players_file_name, X)
        np.save(results_file_name, Y)
        
    
create_world_cup_match_dataset()

Processing batch #0 which goes from row 0 to 100
Processing batch #1 which goes from row 100 to 200
Processing batch #2 which goes from row 200 to 300
Processing batch #3 which goes from row 300 to 400
Processing batch #4 which goes from row 400 to 500
Processing batch #5 which goes from row 500 to 600
Processing batch #6 which goes from row 600 to 700


In [153]:
X = np.load("players_batch_6_2026-05-25 19:22:48.255657.npy")
print(X.shape)
Y = np.load("results_batch_6_2026-05-25 19:22:48.255680.npy")
print(Y.shape)

(700, 22, 9)
(700, 2, 1)


In [77]:
Y = np.load('results_batch_6_2026-05-25 16:28:40.426796.npy')
print(Y)

[[[0.]
  [0.]]

 [[2.]
  [0.]]

 [[1.]
  [0.]]

 ...

 [[2.]
  [1.]]

 [[2.]
  [0.]]

 [[4.]
  [2.]]]
